# Objective 2.2 — Predicting a Team's Goals in a Single Match

## Goal

Build and evaluate a linear regression model that predicts the **number of
goals scored by a team** in a single 2026 FIFA World Cup match, using a
dataset of **208 rows** (2 rows per match × 104 matches — one row per team
per match), and **8 explanatory variables that are all knowable before
kickoff**.

| # | Variable | Description |
|---|----------|--------------|
| 1 | `team_rank` | Team's own FIFA ranking position |
| 2 | `opp_rank` | Opponent's FIFA ranking position |
| 3 | `team_value` | Team's squad market value (EUR m) |
| 4 | `opp_value` | Opponent's squad market value (EUR m) |
| 5 | `team_age` | Team's squad average age |
| 6 | `is_host` | 1 if the team is a co-host nation, else 0 |
| 7 | `rest_days` | Days since the team's previous tournament match |
| 8 | `knockout` | 1 if the match is in the knockout stage, else 0 |

None of these variables depend on anything that happens *during* the match
itself, satisfying the "available before the match" requirement. This
model is related to, but distinct from, the 2.1 goal-difference model: it
predicts an absolute, team-specific goal count rather than a relative,
match-level difference, and it is estimated on the finer team-match grain
(208 rows) rather than the match grain (104 rows).

In [ ]:
import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from data_prep import load_matches, load_teams, build_team_match_dataset

pd.set_option("display.max_columns", None)
np.random.seed(42)

matches = load_matches()
teams = load_teams()
reg_data = build_team_match_dataset(matches, teams)
print(reg_data.shape)
reg_data.head()

## 1. Sanity checks & exploratory data analysis

In [ ]:
assert reg_data.shape[0] == 208, f"Expected 208 rows, got {reg_data.shape[0]}"
feature_cols = [
    "team_rank", "opp_rank", "team_value", "opp_value",
    "team_age", "is_host", "rest_days", "knockout",
]
assert len(feature_cols) == 8
print(reg_data[feature_cols + ["goals"]].isna().sum())
reg_data[feature_cols + ["goals"]].describe()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(reg_data["goals"], bins=range(0, int(reg_data["goals"].max()) + 2), edgecolor="white")
ax.set_title("Distribution of goals scored by a team in a match, n=208")
ax.set_xlabel("Goals")
plt.tight_layout()
plt.savefig("../report/figs/reg2_target_hist.png", dpi=120)
plt.show()

corr = reg_data[feature_cols + ["goals"]].corr()
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
plt.title("Correlation matrix")
plt.tight_layout()
plt.savefig("../report/figs/reg2_corr.png", dpi=120)
plt.show()
corr["goals"].sort_values(ascending=False)

## 2. Multicollinearity check (VIF)

Variance Inflation Factors above ~5–10 would indicate problematic
redundancy among explanatory variables that could inflate coefficient
standard errors.

In [ ]:
X_all = sm.add_constant(reg_data[feature_cols])
vif = pd.DataFrame(
    {
        "feature": X_all.columns,
        "VIF": [variance_inflation_factor(X_all.values, i) for i in range(X_all.shape[1])],
    }
)
vif

## 3. Train/test split

We hold out 20% of team-match rows (≈42 rows) as a test set for
out-of-sample evaluation; note the two rows belonging to the same match
are independent observations from the model's perspective (different
teams, different feature values), so a plain random split is appropriate
here rather than a match-level grouped split.

In [ ]:
X = reg_data[feature_cols]
y = reg_data["goals"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
print("Train:", X_train.shape, " Test:", X_test.shape)

## 4. Fit and interpret the OLS model (training set)

In [ ]:
X_train_sm = sm.add_constant(X_train)
ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())

## 5. Residual diagnostics

In [ ]:
fitted = ols_model.fittedvalues
resid = ols_model.resid

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].scatter(fitted, resid, alpha=0.7)
ax[0].axhline(0, color="red", linestyle="--")
ax[0].set_xlabel("Fitted values")
ax[0].set_ylabel("Residuals")
ax[0].set_title("Residuals vs. fitted")

sm.qqplot(resid, line="45", fit=True, ax=ax[1])
ax[1].set_title("Q-Q plot of residuals")

ax[2].hist(resid, bins=15, edgecolor="white")
ax[2].set_title("Residual distribution")
plt.tight_layout()
plt.savefig("../report/figs/reg2_diagnostics.png", dpi=120)
plt.show()

from scipy import stats as sstats
shapiro_stat, shapiro_p = sstats.shapiro(resid)
print(f"Shapiro-Wilk normality test on residuals: stat={shapiro_stat:.4f}, p={shapiro_p:.4f}")

## 6. Out-of-sample evaluation (held-out test rows)

In [ ]:
X_test_sm = sm.add_constant(X_test, has_constant="add")[X_train_sm.columns]
y_pred = ols_model.predict(X_test_sm)

rmse = mean_squared_error(y_test, y_pred) ** 0.5
mae = mean_absolute_error(y_test, y_pred)
r2_test = r2_score(y_test, y_pred)

n_train, k = X_train.shape[0], X_train.shape[1]
adj_r2_train = 1 - (1 - ols_model.rsquared) * (n_train - 1) / (n_train - k - 1)

print(f"Training R-squared:          {ols_model.rsquared:.4f}")
print(f"Training adjusted R-squared: {adj_r2_train:.4f}")
print(f"Test R-squared:              {r2_test:.4f}")
print(f"Test RMSE:                   {rmse:.3f} goals")
print(f"Test MAE:                    {mae:.3f} goals")

fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test, y_pred, alpha=0.75)
lims = [min(y_test.min(), y_pred.min()) - 0.5, max(y_test.max(), y_pred.max()) + 0.5]
ax.plot(lims, lims, "r--")
ax.set_xlabel("Actual goals scored")
ax.set_ylabel("Predicted goals scored")
ax.set_title("Test set: predicted vs. actual")
plt.tight_layout()
plt.savefig("../report/figs/reg2_pred_vs_actual.png", dpi=120)
plt.show()

## 7. Conclusion

*(Auto-filled after running the cells above with real data. Summarize:
which of the 8 explanatory variables are statistically significant
predictors of a team's goals scored (p < 0.05) and the sign/interpretation
of their coefficients, the model's overall fit (R², adjusted R²) and
out-of-sample RMSE/MAE, how it compares with the 2.1 goal-difference model,
and any diagnostic concerns such as residual non-normality or
heteroscedasticity (goals being a small non-negative count variable, note
whether a Poisson/negative-binomial model might be a more natural
alternative to OLS in future work).)*